# 07 可视化模块 (core.viz)

基于 `hscredit.xlsx` 真实信贷数据，演示所有图表。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import (init_setting, WOEEncoder, LogisticRegression, OptimalBinning,
    bin_plot, corr_plot, ks_plot, hist_plot, psi_plot, distribution_plot,
    roc_plot, pr_plot, lift_plot, gain_plot, confusion_matrix_plot, calibration_plot,
    score_dist_plot, score_bin_plot, plot_weights,
    bin_trend_plot, overdues_bin_plot,
    variable_iv_plot, variable_missing_badrate_plot,
    score_ks_plot, score_lift_plot, score_approval_badrate_curve,
    feature_trend_by_time, feature_drift_comparison, feature_cross_heatmap,
    threshold_analysis_plot, strategy_compare_plot, vintage_plot,
    feature_importance_plot, approval_rate_trend_plot, bad_rate_trend_plot,
    segment_scorecard_comparison,
)
from hscredit.core.metrics import ks, auc
init_setting()

df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D', origin='1899-12-30')
df['loan_month'] = df['loan_date'].dt.to_period('M').astype(str)
target = 'FPD15'
features = ['bj_qy24','mobtech80','bairong_v1','lender_count_12m','overdue_loan_m1_count_12m',
    'loan_count_12m','loan_amt_sum_12m','network_loan_lender_count',
    'last_performance_days','lender_inquiry_count_3m']
X = df[features].fillna(-999); y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
woe = WOEEncoder(max_n_bins=8)
X_woe_tr = woe.fit_transform(X_tr, y_tr)
X_woe_te = woe.transform(X_te)
lr = LogisticRegression(C=0.1, max_iter=1000, calculate_stats=True); lr.fit(X_woe_tr, y_tr)
y_prob_te = lr.predict_proba(X_woe_te)[:,1]
y_prob_tr = lr.predict_proba(X_woe_tr)[:,1]
print('模型准备完成, KS:', round(ks(y_te,y_prob_te),4))

## 1. bin_plot — 特征分箱图

In [ ]:
b = OptimalBinning(method='mdlp', max_n_bins=8)
b.fit(df[['bj_qy24']].fillna(-999), y)
fig = bin_plot(b.bin_tables_['bj_qy24'], desc='bj_qy24 征信评分')
plt.show()

In [ ]:
# 直接传入DataFrame+特征名
fig = bin_plot(df.fillna(-999), feature='overdue_loan_m1_count_12m', target=target,
    n_bins=6, desc='12个月M1逾期次数')
plt.show()

## 2. corr_plot — 相关性热力图

In [ ]:
fig = corr_plot(df[features].fillna(-999), figure_size=(12,8), mask=True)
plt.show()

## 3. ks_plot — KS & ROC 曲线

In [ ]:
fig = ks_plot(y_prob_te, y_te, title='测试集 KS/ROC')
plt.show()

## 4. hist_plot — 分布直方图

In [ ]:
fig = hist_plot(df['bj_qy24'].dropna(), y_true=y[df['bj_qy24'].notna()], desc='bj_qy24')
plt.show()

## 5. distribution_plot — 时间分布

In [ ]:
fig = distribution_plot(df, date='loan_date', target=target, freq='M')
plt.show()

## 6. roc_plot / pr_plot / lift_plot / gain_plot

In [ ]:
fig = roc_plot(y_te.values, y_prob_te); plt.show()
fig = pr_plot(y_te.values, y_prob_te); plt.show()
fig = lift_plot(y_te.values, y_prob_te); plt.show()
fig = gain_plot(y_te.values, y_prob_te); plt.show()

## 7. score_plots — 评分分析

In [ ]:
from hscredit.core.models.probability_to_score import probability_to_score
scores_te = probability_to_score(y_prob_te, pdo=20, score=600, odds=1/19)
scores_tr = probability_to_score(y_prob_tr, pdo=20, score=600, odds=1/19)
fig = score_dist_plot(scores_te, y_te.values); plt.show()
fig = score_bin_plot(scores_te, y_te.values, n_bins=10); plt.show()

In [ ]:
fig = score_ks_plot(datasets={'训练集':(y_tr.values,scores_tr),'测试集':(y_te.values,scores_te)})
plt.show()
fig = score_lift_plot(y_te.values, scores_te); plt.show()
fig = score_approval_badrate_curve(y_te.values, scores_te); plt.show()

## 8. threshold_analysis_plot — 阈值分析

In [ ]:
fig = threshold_analysis_plot(y_te.values, y_prob_te, thresholds=np.linspace(0.05,0.5,20))
plt.show()

## 9. variable_iv_plot / variable_missing_badrate_plot

In [ ]:
fig = variable_iv_plot(df.fillna(-999), features, target, top_n=10); plt.show()
fig = variable_missing_badrate_plot(df, features, target); plt.show()

## 10. feature_trend_by_time — 特征时序趋势

In [ ]:
fig = feature_trend_by_time(df.fillna(-999), 'bj_qy24', 'loan_date', stat='mean', freq='M')
plt.show()
fig = feature_trend_by_time(df.fillna(-999), 'bj_qy24', 'loan_date', target=target, stat='badrate', freq='M')
plt.show()

## 11. feature_drift_comparison

In [ ]:
fig = feature_drift_comparison(X_tr, X_te, features, top_n=10)
plt.show()

## 12. feature_cross_heatmap

In [ ]:
fig = feature_cross_heatmap(df.fillna(-999), 'bj_qy24', 'overdue_loan_m1_count_12m', target, stat='badrate', n_bins=5)
plt.show()

## 13. plot_weights — 系数图

In [ ]:
fig = plot_weights(lr); plt.show()

## 14. bin_trend_plot — 分箱趋势图

In [ ]:
fig = bin_trend_plot(df.fillna(-999), feature='bj_qy24', target=target,
    date_col='loan_date', date_freq='Q', n_bins=6)
plt.show()